# Build Evaluation Summaries

Utility notebook for consolidating CHAIR and POPE evaluation metrics into summary JSONL files that downstream tooling can consume.

In [ ]:
import json
from pathlib import Path
from typing import Dict, List, Tuple


def resolve_repo_root() -> Path:
    """Locate the repository root that contains the opera_log directory."""
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "opera_log").exists():
            return candidate.resolve()
    raise FileNotFoundError("Cannot locate opera_log directory from current working directory.")


def parse_method_variant(name: str) -> Tuple[str, str]:
    """Split a method directory label into base method and variant."""
    if "-" in name:
        base, variant = name.split("-", 1)
    else:
        base, variant = name, "baseline"
    return base, variant


In [ ]:
def collect_chair_records(chair_root: Path) -> List[Dict]:
    records: List[Dict] = []
    for provider_dir in sorted(p for p in chair_root.iterdir() if p.is_dir()):
        provider = provider_dir.name
        for model_dir in sorted(p for p in provider_dir.iterdir() if p.is_dir()):
            model = model_dir.name
            for method_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
                method_label = method_dir.name
                method_base, method_variant = parse_method_variant(method_label)
                for metric_path in sorted(method_dir.glob("metric*.json")):
                    if metric_path.name == "metric.json":
                        metric_variant = "default"
                    elif "metric_" in metric_path.stem:
                        metric_variant = metric_path.stem.split("metric_", 1)[1]
                    else:
                        metric_variant = metric_path.stem.replace("metric", "").strip("_") or "custom"
                    try:
                        data = json.loads(metric_path.read_text())
                    except json.JSONDecodeError as exc:
                        raise ValueError(f"Failed parsing {metric_path}") from exc
                    overall = data.get("overall_metrics") or {
                        k: v for k, v in data.items() if isinstance(v, (int, float))
                    }
                    record = {
                        "dataset": "chair",
                        "provider": provider,
                        "model": model,
                        "method": method_base,
                        "variant": method_variant,
                        "metric_variant": metric_variant,
                        "source": str(metric_path.relative_to(chair_root.parent.parent)),
                        **overall,
                    }
                    records.append(record)
    return records


def collect_pope_records(pope_root: Path) -> List[Dict]:
    type_alias_map = {"adv": "adversarial", "pop": "popular", "rand": "random"}
    records: List[Dict] = []
    metric_globs = ("*_metric.json", "*_metric.jsonl")
    for provider_dir in sorted(p for p in pope_root.iterdir() if p.is_dir()):
        provider = provider_dir.name
        for model_dir in sorted(p for p in provider_dir.iterdir() if p.is_dir()):
            model = model_dir.name
            for type_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
                type_alias = type_dir.name
                pope_type = type_alias_map.get(type_alias)
                if pope_type is None:
                    continue
                for method_dir in sorted(p for p in type_dir.iterdir() if p.is_dir()):
                    method_label = method_dir.name
                    method_base, method_variant = parse_method_variant(method_label)
                    metric_files: List[Path] = []
                    for pattern in metric_globs:
                        metric_files.extend(method_dir.glob(pattern))
                    for metric_path in sorted(metric_files):
                        try:
                            data = json.loads(metric_path.read_text())
                        except json.JSONDecodeError as exc:
                            raise ValueError(f"Failed parsing {metric_path}") from exc
                        report = data.get("Report", {}) if isinstance(data.get("Report"), dict) else {}
                        macro = report.get("macro avg", {}) if isinstance(report, dict) else {}
                        weighted = report.get("weighted avg", {}) if isinstance(report, dict) else {}
                        record = {
                            "dataset": "pope",
                            "provider": provider,
                            "model": model,
                            "pope_type": data.get("POPE_Type", pope_type),
                            "type_alias": type_alias,
                            "method": method_base,
                            "variant": method_variant,
                            "metric_variant": "default",
                            "source": str(metric_path.relative_to(pope_root.parent.parent)),
                            "accuracy": data.get("Accuracy"),
                            "tp": data.get("TP"),
                            "fp": data.get("FP"),
                            "fn": data.get("FN"),
                            "tn": data.get("TN"),
                            "samples": data.get("TotalSamples"),
                            "f1_macro": macro.get("f1-score"),
                            "precision_macro": macro.get("precision"),
                            "recall_macro": macro.get("recall"),
                            "f1_weighted": weighted.get("f1-score"),
                            "precision_weighted": weighted.get("precision"),
                            "recall_weighted": weighted.get("recall"),
                        }
                        records.append(record)
    return records


In [ ]:
if __name__ == "__main__":
    REPO_ROOT = resolve_repo_root()
    CHAIR_ROOT = REPO_ROOT / "opera_log" / "chair_eval_results"
    POPE_ROOT = REPO_ROOT / "opera_log" / "pope_eval_results"
    OUTPUT_DIR = REPO_ROOT / "scripts" / "opera_log" / "summary_jsonl"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    chair_records = collect_chair_records(CHAIR_ROOT)
    pope_records = collect_pope_records(POPE_ROOT)

    chair_records.sort(key=lambda r: (r["provider"], r["model"], r["method"], r["variant"], r["metric_variant"], r["source"]))
    pope_records.sort(key=lambda r: (r["provider"], r["model"], r.get("pope_type"), r["method"], r["variant"], r["source"]))

    chair_path = OUTPUT_DIR / "chair_summary.jsonl"
    pope_path = OUTPUT_DIR / "pope_summary.jsonl"

    with chair_path.open("w") as f:
        for record in chair_records:
            f.write(json.dumps(record) + "
")

    with pope_path.open("w") as f:
        for record in pope_records:
            f.write(json.dumps(record) + "
")

    print(f"✅ Saved {len(chair_records)} CHAIR runs → {chair_path.relative_to(REPO_ROOT)}")
    print(f"✅ Saved {len(pope_records)} POPE runs → {pope_path.relative_to(REPO_ROOT)}")
